In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas correctamente.")

# Cargar dataset
daily = pd.read_csv("../data/processed/eurusd_con_sentimiento.csv",
                    index_col=0, parse_dates=True)

print(f"Dataset: {daily.shape}")

Librerías cargadas correctamente.
Dataset: (4031, 40)


In [5]:
features_ppo = [
    'momentum_5', 'momentum_10',
    'volatility_20', 'rsi_14', 'bb_position',
    'macd', 'sentiment_score', 'regime', 'trend_20_50', 'atr_14'
]

data_clean = daily[features_ppo + ['returns']].dropna().reset_index(drop=True)
env = KronosTradingEnv(data_clean, features_ppo)
check_env(env, warn=True)
print("Entorno verificado correctamente.")
print(f"Observaciones: {env.observation_space.shape}")
print(f"Acciones: {env.action_space.n} (HOLD, BUY, SELL)")

Entorno verificado correctamente.
Observaciones: (13,)
Acciones: 3 (HOLD, BUY, SELL)


In [7]:
print("Entrenando agente PPO...")

modelo_ppo = PPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=512,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    verbose=0
)

modelo_ppo.learn(total_timesteps=50000)
print("Entrenamiento completado.")

obs, _ = env.reset()
rewards_totales = []
acciones = []
done = False

while not done:
    action, _ = modelo_ppo.predict(obs, deterministic=True)
    obs, reward, done, _, _ = env.step(action)
    rewards_totales.append(reward)
    acciones.append(action)

retorno_total = sum(rewards_totales)
acciones = np.array(acciones)
print(f"Retorno acumulado: {retorno_total:.4f}")
print(f"HOLD: {(acciones==0).sum()} ({(acciones==0).mean()*100:.1f}%)")
print(f"BUY:  {(acciones==1).sum()} ({(acciones==1).mean()*100:.1f}%)")
print(f"SELL: {(acciones==2).sum()} ({(acciones==2).mean()*100:.1f}%)")

Entrenando agente PPO...
Entrenamiento completado.
Retorno acumulado: -3.3630
HOLD: 0 (0.0%)
BUY:  3980 (99.3%)
SELL: 30 (0.7%)


In [8]:
# Resetear entorno y reentrenar con más timesteps
env2 = KronosTradingEnv(data_clean, features_ppo)

modelo_ppo2 = PPO(
    "MlpPolicy",
    env2,
    learning_rate=1e-4,
    n_steps=1024,
    batch_size=128,
    n_epochs=10,
    gamma=0.95,
    ent_coef=0.01,  # exploración
    verbose=0
)

print("Entrenando PPO con más exploración...")
modelo_ppo2.learn(total_timesteps=200000)
print("Completado.")

obs, _ = env2.reset()
rewards_totales2 = []
acciones2 = []
done = False

while not done:
    action, _ = modelo_ppo2.predict(obs, deterministic=True)
    obs, reward, done, _, _ = env2.step(action)
    rewards_totales2.append(reward)
    acciones2.append(action)

retorno2 = sum(rewards_totales2)
acciones2 = np.array(acciones2)
print(f"\nRetorno acumulado: {retorno2:.4f}")
print(f"HOLD: {(acciones2==0).sum()} ({(acciones2==0).mean()*100:.1f}%)")
print(f"BUY:  {(acciones2==1).sum()} ({(acciones2==1).mean()*100:.1f}%)")
print(f"SELL: {(acciones2==2).sum()} ({(acciones2==2).mean()*100:.1f}%)")

Entrenando PPO con más exploración...
Completado.

Retorno acumulado: 0.0400
HOLD: 0 (0.0%)
BUY:  81 (2.0%)
SELL: 3929 (98.0%)


In [9]:
# Guardar modelo PPO
modelo_ppo2.save("../data/processed/ppo_agent")
print("Agente PPO guardado.")

print(f"\nResumen Fase 9:")
print(f"  Entorno: KronosTradingEnv")
print(f"  Observaciones: 13 features")
print(f"  Acciones: HOLD, BUY, SELL")
print(f"  Timesteps: 200,000")
print(f"  Retorno acumulado: 0.04")
print(f"  Estado: prototipo funcional")
print(f"  Pendiente: reward shaping y más timesteps")

Agente PPO guardado.

Resumen Fase 9:
  Entorno: KronosTradingEnv
  Observaciones: 13 features
  Acciones: HOLD, BUY, SELL
  Timesteps: 200,000
  Retorno acumulado: 0.04
  Estado: prototipo funcional
  Pendiente: reward shaping y más timesteps


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import EvalCallback
import warnings
warnings.filterwarnings('ignore')

daily = pd.read_csv("../data/processed/eurusd_con_sentimiento.csv",
                    index_col=0, parse_dates=True)

print(f"Dataset: {daily.shape}")
print("Listo.")

Dataset: (4031, 40)
Listo.


In [2]:
class KronosTradingEnvV2(gym.Env):
    def __init__(self, data, features, window=20):
        super().__init__()
        self.data = data.reset_index(drop=True)
        self.features = features
        self.window = window
        self.n_features = len(features) + 4
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(self.n_features,), dtype=np.float32
        )
        self.reset()

    def reset(self, seed=None):
        super().reset(seed=seed)
        self.current_step = self.window
        self.position = 0
        self.pnl = 0.0
        self.tiempo_en_pos = 0
        self.trades = 0
        self.wins = 0
        self.peak_pnl = 0.0
        return self._get_obs(), {}

    def _get_obs(self):
        row = self.data[self.features].iloc[self.current_step]
        obs = np.array(
            row.values.tolist() + [self.position, self.pnl, 
                                    self.tiempo_en_pos, self.trades],
            dtype=np.float32
        )
        return np.nan_to_num(obs, nan=0.0)

    def step(self, action):
        ret = self.data['returns'].iloc[self.current_step]
        
        # Reward base — retorno de la posición
        reward = self.position * ret * 100  # escalar para gradientes
        
        # Penalización por cambio de posición (costo transacción)
        prev_position = self.position
        if action == 1:
            self.position = 1
        elif action == 2:
            self.position = -1
        else:
            self.position = 0
            
        if self.position != prev_position:
            reward -= 0.05  # costo de transacción
            self.trades += 1
        
        # Premio por wins consecutivos
        if reward > 0:
            self.wins += 1
            reward *= 1.1  # bonus por ganar
        
        # Penalización por drawdown
        self.pnl += reward
        if self.pnl > self.peak_pnl:
            self.peak_pnl = self.pnl
        drawdown = (self.pnl - self.peak_pnl) / (abs(self.peak_pnl) + 1e-8)
        if drawdown < -0.1:
            reward -= 0.5
        
        # Penalización por overtrading
        if self.trades > self.current_step * 0.3:
            reward -= 0.1
        
        self.tiempo_en_pos = self.tiempo_en_pos + 1 if self.position != 0 else 0
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1
        return self._get_obs(), reward, done, False, {}

features_ppo = [
    'momentum_5', 'momentum_10',
    'volatility_20', 'rsi_14', 'bb_position',
    'macd', 'sentiment_score', 'regime', 'trend_20_50', 'atr_14'
]

data_clean = daily[features_ppo + ['returns']].dropna().reset_index(drop=True)

# Split train/eval
split = int(len(data_clean) * 0.8)
data_train = data_clean.iloc[:split].reset_index(drop=True)
data_eval = data_clean.iloc[split:].reset_index(drop=True)

env_train = KronosTradingEnvV2(data_train, features_ppo)
env_eval = KronosTradingEnvV2(data_eval, features_ppo)

check_env(env_train, warn=True)
print("Entorno V2 verificado.")
print(f"Train: {len(data_train)} | Eval: {len(data_eval)}")

Entorno V2 verificado.
Train: 3224 | Eval: 807


In [3]:
print("Entrenando PPO V2...")

modelo_ppo_v2 = PPO(
    "MlpPolicy",
    env_train,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=128,
    n_epochs=15,
    gamma=0.99,
    ent_coef=0.005,
    clip_range=0.2,
    verbose=0,
    policy_kwargs=dict(net_arch=[128, 128, 64])
)

modelo_ppo_v2.learn(total_timesteps=300000)
print("Entrenamiento completado.")

# Evaluar en datos de eval
obs, _ = env_eval.reset()
rewards_eval = []
acciones_eval = []
done = False

while not done:
    action, _ = modelo_ppo_v2.predict(obs, deterministic=True)
    obs, reward, done, _, _ = env_eval.step(action)
    rewards_eval.append(reward)
    acciones_eval.append(int(action))

acciones_eval = np.array(acciones_eval)
print(f"\nEvaluación PPO V2:")
print(f"  Retorno acumulado: {sum(rewards_eval):.4f}")
print(f"  HOLD: {(acciones_eval==0).sum()} ({(acciones_eval==0).mean()*100:.1f}%)")
print(f"  BUY:  {(acciones_eval==1).sum()} ({(acciones_eval==1).mean()*100:.1f}%)")
print(f"  SELL: {(acciones_eval==2).sum()} ({(acciones_eval==2).mean()*100:.1f}%)")

Entrenando PPO V2...
Entrenamiento completado.

Evaluación PPO V2:
  Retorno acumulado: 0.0000
  HOLD: 786 (100.0%)
  BUY:  0 (0.0%)
  SELL: 0 (0.0%)


In [4]:
class KronosTradingEnvV3(gym.Env):
    def __init__(self, data, features, window=20):
        super().__init__()
        self.data = data.reset_index(drop=True)
        self.features = features
        self.window = window
        self.n_features = len(features) + 3
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(self.n_features,), dtype=np.float32
        )
        self.reset()

    def reset(self, seed=None):
        super().reset(seed=seed)
        self.current_step = self.window
        self.position = 0
        self.pnl = 0.0
        self.tiempo_en_pos = 0
        return self._get_obs(), {}

    def _get_obs(self):
        row = self.data[self.features].iloc[self.current_step]
        obs = np.array(
            row.values.tolist() + [self.position, self.pnl, self.tiempo_en_pos],
            dtype=np.float32
        )
        return np.nan_to_num(obs, nan=0.0)

    def step(self, action):
        ret = self.data['returns'].iloc[self.current_step]
        prev_position = self.position
        
        if action == 1:
            self.position = 1
        elif action == 2:
            self.position = -1
        else:
            self.position = 0

        # Reward simple y bien escalado
        reward = float(self.position * ret * 1000)
        
        # Penalización pequeña por transacción
        if self.position != prev_position and self.position != 0:
            reward -= 0.1
        
        # Penalización leve por HOLD prolongado
        if self.position == 0:
            reward -= 0.01

        self.pnl += reward
        self.tiempo_en_pos = self.tiempo_en_pos + 1 if self.position != 0 else 0
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1
        return self._get_obs(), reward, done, False, {}

env_v3 = KronosTradingEnvV3(data_train, features_ppo)
check_env(env_v3, warn=True)

modelo_ppo_v3 = PPO(
    "MlpPolicy",
    env_v3,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    gamma=0.95,
    ent_coef=0.02,
    verbose=0
)

print("Entrenando PPO V3...")
modelo_ppo_v3.learn(total_timesteps=300000)
print("Completado.")

obs, _ = KronosTradingEnvV3(data_eval, features_ppo).reset()
env_test = KronosTradingEnvV3(data_eval, features_ppo)
obs, _ = env_test.reset()
rewards_v3 = []
acciones_v3 = []
done = False

while not done:
    action, _ = modelo_ppo_v3.predict(obs, deterministic=True)
    obs, reward, done, _, _ = env_test.step(action)
    rewards_v3.append(reward)
    acciones_v3.append(int(action))

acciones_v3 = np.array(acciones_v3)
print(f"\nEvaluación PPO V3:")
print(f"  Retorno acumulado: {sum(rewards_v3):.2f}")
print(f"  HOLD: {(acciones_v3==0).sum()} ({(acciones_v3==0).mean()*100:.1f}%)")
print(f"  BUY:  {(acciones_v3==1).sum()} ({(acciones_v3==1).mean()*100:.1f}%)")
print(f"  SELL: {(acciones_v3==2).sum()} ({(acciones_v3==2).mean()*100:.1f}%)")

Entrenando PPO V3...
Completado.

Evaluación PPO V3:
  Retorno acumulado: 4.64
  HOLD: 0 (0.0%)
  BUY:  25 (3.2%)
  SELL: 761 (96.8%)


In [5]:
modelo_ppo_v3.save("../data/processed/ppo_agent_v3")
print("PPO V3 guardado.")
print(f"\nResumen PPO V3:")
print(f"  Timesteps: 300,000")
print(f"  Retorno acumulado eval: 4.64")
print(f"  Distribución: 3.2% BUY, 96.8% SELL")
print(f"  Diagnóstico: aprende tendencia bajista correctamente")
print(f"  Mejora vs V1: retorno positivo (+4.64 vs -3.36)")

PPO V3 guardado.

Resumen PPO V3:
  Timesteps: 300,000
  Retorno acumulado eval: 4.64
  Distribución: 3.2% BUY, 96.8% SELL
  Diagnóstico: aprende tendencia bajista correctamente
  Mejora vs V1: retorno positivo (+4.64 vs -3.36)
